# Whose Evidence Does the Model Believe? — pilot run

Jacobian-lens audit of source deference (AAAI FSS 2026, "Aligning with Whom?").
Pilot: English + NE_GS pair (80 prompts, ~10 min on a free T4).

**Before running:** Runtime → Change runtime type → **GPU (T4)**.

In [ ]:
!git clone --depth 1 https://github.com/anthropics/jacobian-lens.git
!git clone --depth 1 https://github.com/mleyvaz/jspace-epistemic-lens.git
%cd jacobian-lens
!pip install -q -e .
!nvidia-smi | head -12

In [ ]:
import torch, transformers, jlens
jlens.configure_logging()

MODEL_NAME = "Qwen/Qwen3.5-4B"
hf_model = transformers.AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.bfloat16).cuda()
tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_NAME)
model = jlens.from_hf(hf_model, tokenizer)

lens = jlens.JacobianLens.from_pretrained(
    "neuronpedia/jacobian-lens",
    filename="qwen3.5-4b/jlens/Salesforce-wikitext/Qwen3.5-4B_jacobian_lens_n1000.pt",
    revision="qwen-n1000",
)
print("layers:", model.n_layers)

In [ ]:
import json, sys
sys.path.append("/content/jspace-epistemic-lens/src")
import jspace
lexids = jspace.build_lexicons(tokenizer)

with open("/content/jspace-epistemic-lens/experiments/source_deference/stimuli_source_deference.json") as f:
    ALL = json.load(f)

# ---- PILOT: English + NE_GS pair only (80 prompts) ----
STIM = [r for r in ALL if r["lang"] == "en" and r["pair"] == "NE_GS"]
print("pilot stimuli:", len(STIM))

In [ ]:
import pandas as pd

LAYERS = list(range(18, 31))
rows = []
for k, r in enumerate(STIM):
    jl, ml, _ = lens.apply(model, r["prompt"], layers=LAYERS, positions=[-2])
    for L in LAYERS:
        m, ent = jspace.log_masses(jl[L][0], lexids)
        rows.append(dict(id=r["id"], fact_id=r["fact_id"], pair=r["pair"],
                         aff_type=r["aff_type"], den_type=r["den_type"],
                         aff_first=r["aff_first"], layer=L, ent=ent,
                         log_T=m["T"], log_I=m["I"], log_F=m["F"]))
    if (k + 1) % 20 == 0:
        print(f"{k+1}/{len(STIM)}")
df = pd.DataFrame(rows)
df["B"] = df.log_F - df.log_T   # verdict balance: >0 leans false, <0 leans true
df.to_csv("source_deference_pilot.csv", index=False)
print("saved:", df.shape)

In [ ]:
import numpy as np
from scipy import stats as st

CLEAN = [L for L in LAYERS if L >= 22]
d = df[df.layer.isin(CLEAN)]
# per-stimulus mean balance over clean layers
perstim = d.groupby(["id", "aff_type", "aff_first"]).B.mean().reset_index()

print("=" * 66)
print("SANITY 1 - Does the internal verdict track TRUTH at all?")
print("  (every claim is TRUE, so truth-tracking => mean B < 0)")
print(f"  overall mean B = {perstim.B.mean():+.3f}")
print()
print("=" * 66)
print("SANITY 2 - RECENCY (nuisance): B when denier is LAST vs FIRST")
# denier last  == affirmer first (aff_first True)
bl = perstim[perstim.aff_first == True].B     # denier second/last
bf = perstim[perstim.aff_first == False].B    # denier first
R = bf.mean() - bl.mean()
u, p = st.mannwhitneyu(bf, bl)
print(f"  B(denier first) = {bf.mean():+.3f} | B(denier last) = {bl.mean():+.3f}")
print(f"  recency index R = {R:+.3f}  (p={p:.1e})")
print()
print("=" * 66)
print("KEY - SOURCE-SIDE effect (prestige+geography, order-averaged)")
print("  aff=NE : elite affirms the TRUE claim (elite on truth's side)")
print("  aff=GS : elite DENIES the true claim (elite on falsehood's side)")
bNE = perstim[perstim.aff_type == "NE"].B
bGS = perstim[perstim.aff_type == "GS"].B
S = bGS.mean() - bNE.mean()
u, p = st.mannwhitneyu(bGS, bNE, alternative="greater")
print(f"  B(NE affirms) = {bNE.mean():+.3f} | B(GS affirms / NE denies) = {bGS.mean():+.3f}")
print(f"  source-side index S = {S:+.3f}  (p={p:.1e})")
print()
print("  READ: S>0 and significant => the model leans FALSE when the elite")
print("  denies / GS affirms => genuine source sensitivity beyond recency =>")
print("  paper has legs at 4B. S~0 => 4B is recency-only => push to larger")
print("  models (interesting null either way). Full run (WK anchor) then")
print("  separates prestige-over-truth from geographic discount.")

from google.colab import files
files.download("source_deference_pilot.csv")